# Notebook 4: Multi-Session Training and Leave-One-Session-Out Evaluation

This notebook implements comprehensive multi-session biometric evaluation and leave-one-session-out (LOSO) testing:
1. **Experiment A (Multi-Session Training - LOSO)**:
   - Trains biometric models on three sessions and tests on the fourth held-out session.
   - Evaluates all 4 rotation combinations:
     - Train: Sessions 1, 2, 3 $\to$ Test: Session 4
     - Train: Sessions 1, 2, 4 $\to$ Test: Session 3
     - Train: Sessions 1, 3, 4 $\to$ Test: Session 2
     - Train: Sessions 2, 3, 4 $\to$ Test: Session 1
2. **Experiment B (Single-Session Training vs Remaining-Session Testing)**:
   - Evaluates the complementary protocol:
     - Train: Session 1 $\to$ Test: Sessions 2, 3, 4
     - Train: Session 2 $\to$ Test: Sessions 1, 3, 4
     - Train: Session 3 $\to$ Test: Sessions 1, 2, 4
     - Train: Session 4 $\to$ Test: Sessions 1, 2, 3
3. **Rigorous Genuine and Impostor Testing**:
   - Genuine test samples are drawn exclusively from held-out session(s).
   - An equal number of impostor samples are drawn from other users in those exact test sessions ($N_{impostor} = N_{genuine}$).
   - Zero data leakage: Feature scalers and models are fitted strictly on training sessions.
4. **Direct Systematic Comparison**:
   - Quantifies the biometric fidelity gain and error rate reduction achieved by accumulating multi-session training data compared to single-session enrollment.
5. **Comprehensive Model Evaluation**: Runs all 8 advanced models (4 ML and 4 DL architectures) across brain regions and montages using strictly spectral-domain features.


In [1]:
# Environment setup, reproducible random seeds, and imports
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
import mne
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
import xgboost as xgb
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    _ = torch.zeros(1, device=DEVICE)

print(f"Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Random Seed   : {RANDOM_SEED}")


Compute Device: cuda (NVIDIA GeForce RTX 3050 Laptop GPU)
Random Seed   : 42


## Section 1: Preprocessing and Session-Indexed Spectral Features

We load the complete EEG dataset across all 40 recording files (5 subjects, 4 sessions each, Closed Eyes and Open Eyes conditions).
Applying the project-wide common cap of 45 epochs per file, exactly 20 spectral-domain features are extracted per channel.
Statistical, time-domain, Hjorth, and entropy metrics are strictly excluded.


In [2]:
# High-performance vectorized spectral feature extractor
_trapezoid = getattr(np, "trapezoid", getattr(np, "trapz", None))

FREQ_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 50.0)
}

def extract_spectral_features_tensor(data, sfreq=250.0):
    """Vectorized extraction of 20 spectral features per electrode channel."""
    n_epochs, n_channels, n_times = data.shape
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(256, n_times), axis=-1)
    
    # 1. Absolute band powers (5)
    bp_list = []
    for (f_lo, f_hi) in FREQ_BANDS.values():
        mask = (freqs >= f_lo) & (freqs <= f_hi)
        p = _trapezoid(psd[:, :, mask], freqs[mask], axis=-1)
        bp_list.append(p)
    bp_arr = np.stack(bp_list, axis=-1)
    
    # 2. Total power (1-50 Hz) (1)
    tot_mask = (freqs >= 1.0) & (freqs <= 50.0)
    total_power = np.expand_dims(_trapezoid(psd[:, :, tot_mask], freqs[tot_mask], axis=-1), axis=-1) + 1e-12
    
    # 3. Relative band powers (5)
    rel_bp = bp_arr / total_power
    
    # 4. Spectral band power ratios (4)
    delta = bp_arr[..., 0] + 1e-12
    theta = bp_arr[..., 1] + 1e-12
    alpha = bp_arr[..., 2] + 1e-12
    beta  = bp_arr[..., 3] + 1e-12
    tbr = np.expand_dims(theta / beta, -1)
    tar = np.expand_dims(theta / alpha, -1)
    abr = np.expand_dims(alpha / beta, -1)
    slow_fast = np.expand_dims((delta + theta) / (alpha + beta), -1)
    ratios = np.concatenate([tbr, tar, abr, slow_fast], axis=-1)
    
    # 5. Spectral distribution & shape metrics (5)
    freqs_sub = freqs[tot_mask]
    psd_sub = psd[:, :, tot_mask]
    p_sum = psd_sub.sum(axis=-1, keepdims=True) + 1e-12
    centroid = (psd_sub * freqs_sub).sum(axis=-1, keepdims=True) / p_sum
    spread = np.sqrt(((freqs_sub - centroid)**2 * psd_sub).sum(axis=-1, keepdims=True) / p_sum)
    
    cum_psd = np.cumsum(psd_sub, axis=-1)
    cum_norm = cum_psd / (cum_psd[..., -1:] + 1e-12)
    idx50 = (cum_norm >= 0.5).argmax(axis=-1, keepdims=True)
    idx95 = (cum_norm >= 0.95).argmax(axis=-1, keepdims=True)
    sef50 = np.take_along_axis(np.broadcast_to(freqs_sub, idx50.shape[:-1] + (len(freqs_sub),)), idx50, axis=-1)
    sef95 = np.take_along_axis(np.broadcast_to(freqs_sub, idx95.shape[:-1] + (len(freqs_sub),)), idx95, axis=-1)
    
    idx_peak = psd_sub.argmax(axis=-1, keepdims=True)
    peak_freq = np.take_along_axis(np.broadcast_to(freqs_sub, idx_peak.shape[:-1] + (len(freqs_sub),)), idx_peak, axis=-1)
    spectral_dist = np.concatenate([centroid, spread, sef50, sef95, peak_freq], axis=-1)
    
    features = np.concatenate([bp_arr, total_power, rel_bp, ratios, spectral_dist], axis=-1)
    return features.astype(np.float32)

dataset_path = Path("../preprocessed_dataset")
if not dataset_path.exists():
    dataset_path = Path("preprocessed_dataset")

fif_files = sorted(list(dataset_path.glob("**/*_desc-preproc_epo.fif")))
EPOCH_CAP = 45

sample_ep = mne.read_epochs(str(fif_files[0]), preload=False, verbose=False)
ALL_CHANNELS = list(sample_ep.ch_names)
CHANNEL_TO_IDX = {ch: i for i, ch in enumerate(ALL_CHANNELS)}

REGIONS = {
    "Frontal": ["Fp1", "Fpz", "Fp2", "AF3", "AF4", "F7", "F5", "F3", "F1", "Fz", "F2", "F4", "F6", "F8"],
    "Central": ["FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6", "C5", "C3", "C1", "Cz", "C2", "C4", "C6"],
    "Temporal": ["T7", "T8", "TP7", "TP8", "CP5", "CP6"],
    "Parietal": ["CP3", "CP1", "CPz", "CP2", "CP4", "P7", "P5", "P3", "P1", "Pz", "P2", "P4", "P6", "P8"],
    "Occipital": ["PO7", "PO5", "PO3", "POz", "PO4", "PO6", "PO8", "O1", "Oz", "O2"],
    "ALL": ALL_CHANNELS
}

print(f"Extracting spectral features for all {len(fif_files)} files (Capped at {EPOCH_CAP} epochs)...")
t0 = time.time()
session_data = {}

for fpath in fif_files:
    fname = fpath.name
    parts = fpath.parts
    sub = [p for p in parts if p.startswith("sub-")][0]
    ses = [p for p in parts if p.startswith("ses-")][0]
    
    ep = mne.read_epochs(str(fpath), preload=True, verbose=False)
    raw_data = ep.get_data()[:EPOCH_CAP]
    sfreq = ep.info['sfreq']
    
    feats = extract_spectral_features_tensor(raw_data, sfreq=sfreq)
    
    key = (sub, ses)
    if key not in session_data:
        session_data[key] = []
    session_data[key].append(feats)

session_cache = {}
for (sub, ses), feat_list in session_data.items():
    session_cache[(sub, ses)] = np.vstack(feat_list)

SUBJECTS = sorted(list(set(sub for (sub, ses) in session_cache.keys())))
SUBJECT_SESSIONS = {
    sub: sorted([ses for (s, ses) in session_cache.keys() if s == sub])
    for sub in SUBJECTS
}

print(f"Spectral features ready in {time.time() - t0:.2f} seconds.")
print(f"Total Subjects: {len(SUBJECTS)} | Sessions per subject: 4 | Samples per session: 90")


Extracting spectral features for all 40 files (Capped at 45 epochs)...


Spectral features ready in 9.66 seconds.
Total Subjects: 5 | Sessions per subject: 4 | Samples per session: 90


## Section 2: Biometric Metric Engine and Model Definitions

We register the eight advanced models:
1. **XGBoost**: Extreme Gradient Boosting
2. **LightGBM**: Fast Leaf-wise Gradient Boosting
3. **SVM (RBF Kernel)**: Support Vector Machine with Radial Basis Function
4. **Extra Trees**: Extremely Randomized Trees Ensemble
5. **Res-MLP**: Residual Multi-Layer Perceptron (GPU)
6. **1D-CNN**: Convolutional Neural Network (GPU)
7. **BiLSTM-Attention**: Bidirectional Recurrent Sequence Model with Self-Attention (GPU)
8. **SpecFormer**: Multi-Head Self-Attention Spectral Transformer (GPU)

Biometric verification metrics: EER (continuous ROC sweep), Accuracy, FAR, FRR, FP, and FN.


In [3]:
# Biometric Metric Calculator
def evaluate_biometric_predictions(y_true, y_scores, threshold=0.5):
    """Compute standard biometric verification metrics: EER, FAR, FRR, FP, FN, Accuracy."""
    y_true = np.asarray(y_true).astype(int)
    y_scores = np.asarray(y_scores).astype(float)
    
    n_gen = int(np.sum(y_true == 1))
    n_imp = int(np.sum(y_true == 0))
    
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    eer_idx = np.nanargmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2.0)
    
    y_pred = (y_scores >= threshold).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    
    accuracy = float((tp + tn) / (n_gen + n_imp)) if (n_gen + n_imp) > 0 else 0.0
    far = float(fp / n_imp) if n_imp > 0 else 0.0
    frr = float(fn / n_gen) if n_gen > 0 else 0.0
    
    return {
        "Accuracy": accuracy,
        "EER": eer,
        "FAR": far,
        "FRR": frr,
        "FP": fp,
        "FN": fn
    }

# Deep Learning Architectures
class ResMLPNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 64)
        self.relu = nn.ReLU()
        self.res1 = nn.Linear(64, 64)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(64, 1)
    def forward(self, x):
        h = self.relu(self.fc1(x))
        h = h + self.drop(self.relu(self.res1(h)))
        return self.head(h).view(-1)

class CNN1DNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten(),
            nn.Linear(32 * 4, 1)
        )
    def forward(self, x):
        x = x.unsqueeze(1)
        return self.net(x).view(-1)

class BiLSTMAttentionNet(nn.Module):
    def __init__(self, in_features, feat_dim=20):
        super().__init__()
        self.feat_dim = feat_dim
        dim = feat_dim if (in_features % feat_dim == 0 and in_features >= feat_dim) else in_features
        self.is_multichannel = (in_features % feat_dim == 0 and in_features >= feat_dim)
        self.lstm = nn.LSTM(dim if self.is_multichannel else 1, 16, batch_first=True, bidirectional=True)
        self.att = nn.Linear(32, 1)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        if self.is_multichannel:
            x_seq = x.view(x.shape[0], -1, self.feat_dim)
        else:
            x_seq = x.unsqueeze(-1)
        out, _ = self.lstm(x_seq)
        weights = torch.softmax(self.att(out), dim=1)
        ctx = torch.sum(out * weights, dim=1)
        return self.fc(ctx).view(-1)

class SpecFormerNet(nn.Module):
    def __init__(self, in_features, d_model=24, nhead=3):
        super().__init__()
        if in_features % 20 == 0 and in_features >= 20:
            self.tok_proj = nn.Linear(20, d_model)
            self.multi_tok = True
        else:
            self.tok_proj = nn.Linear(1, d_model)
            self.multi_tok = False
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=48, batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        if self.multi_tok:
            x_seq = x.view(x.shape[0], -1, 20)
        else:
            x_seq = x.unsqueeze(-1)
        h = self.tok_proj(x_seq)
        enc = self.encoder(h)
        return self.fc(enc.mean(dim=1)).view(-1)

class PyTorchClassifierWrapper:
    def __init__(self, net_cls, epochs=15, lr=3e-3, random_state=42):
        self.net_cls = net_cls
        self.epochs = epochs
        self.lr = lr
        self.random_state = random_state
        self.model = None

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        in_dim = X.shape[1]
        self.model = self.net_cls(in_dim).to(DEVICE)
        self.model.train()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.BCEWithLogitsLoss()
        
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        y_t = torch.tensor(y, dtype=torch.float32, device=DEVICE)
        
        for _ in range(self.epochs):
            optimizer.zero_grad()
            out = self.model(X_t)
            loss = criterion(out, y_t)
            loss.backward()
            optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            logits = self.model(X_t)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return np.column_stack([1.0 - probs, probs])

def get_model_pipeline(model_name, in_features):
    if model_name == "XGBoost":
        return xgb.XGBClassifier(
            n_estimators=60, max_depth=4, learning_rate=0.1,
            eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1
        )
    elif model_name == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=60, max_depth=4, num_leaves=15, learning_rate=0.1,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
        )
    elif model_name == "SVM_RBF":
        return SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=RANDOM_SEED)
    elif model_name == "ExtraTrees":
        return ExtraTreesClassifier(n_estimators=60, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1)
    elif model_name == "Res-MLP":
        return PyTorchClassifierWrapper(ResMLPNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "1D-CNN":
        return PyTorchClassifierWrapper(CNN1DNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "BiLSTM-Attention":
        return PyTorchClassifierWrapper(BiLSTMAttentionNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "SpecFormer":
        return PyTorchClassifierWrapper(SpecFormerNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    else:
        raise ValueError(f"Unknown model: {model_name}")

ALL_MODEL_NAMES = [
    "XGBoost", "LightGBM", "SVM_RBF", "ExtraTrees",
    "Res-MLP", "1D-CNN", "BiLSTM-Attention", "SpecFormer"
]
print("All 8 advanced model architectures verified and registered.")


All 8 advanced model architectures verified and registered.


## Section 3: Experiment A – Multi-Session Training, Leave-One-Session-Out (LOSO)

We implement multi-session enrollment by training on three sessions and testing on the remaining held-out fourth session.
All four rotation combinations are systematically evaluated:
1. **Fold 1**: Train on Sessions 1, 2, 3 $\to$ Test on Session 4
2. **Fold 2**: Train on Sessions 1, 2, 4 $\to$ Test on Session 3
3. **Fold 3**: Train on Sessions 1, 3, 4 $\to$ Test on Session 2
4. **Fold 4**: Train on Sessions 2, 3, 4 $\to$ Test on Session 1

### Balanced Genuine and Impostor Testing:
- Genuine test samples: 90 epochs from held-out session.
- Impostor test samples: exactly 90 epochs drawn from other subjects in that held-out session ($N_{impostor} = N_{genuine} = 90$).
- Zero data leakage between training and testing.


In [4]:
# Execute Experiment A: Train on 3 Sessions, Test on the 4th (LOSO)
def get_session_features(channel_names):
    ch_indices = [CHANNEL_TO_IDX[ch] for ch in channel_names]
    data_dict = {}
    for (sub, ses), raw_feats in session_cache.items():
        data_dict[(sub, ses)] = raw_feats[:, ch_indices, :].reshape(raw_feats.shape[0], -1)
    return data_dict

# Rotation configurations: (train_indices, test_index)
LOSO_FOLDS = [
    ([0, 1, 2], 3),
    ([0, 1, 3], 2),
    ([0, 2, 3], 1),
    ([1, 2, 3], 0)
]

expA_results = []
print(f"Beginning Experiment A: Multi-Session Training (LOSO) across {len(REGIONS)} regions and {len(ALL_MODEL_NAMES)} models...\n")

for region_name, ch_list in REGIONS.items():
    print(f"--- Experiment A: Region {region_name} ({len(ch_list)} channels, {len(ch_list)*20} spectral features) ---")
    data_dict = get_session_features(ch_list)
    
    for fold_idx, (tr_ses_idx_list, te_ses_idx) in enumerate(LOSO_FOLDS, 1):
        fold_metrics = {m: [] for m in ALL_MODEL_NAMES}
        gen_tr_count = 0
        gen_te_count = 0
        imp_te_count = 0
        
        for target_sub in SUBJECTS:
            sub_sessions = SUBJECT_SESSIONS[target_sub]
            tr_sessions = [sub_sessions[i] for i in tr_ses_idx_list]
            te_session = sub_sessions[te_ses_idx]
            
            # Genuine train (3 sessions x 90 = 270 samples) & test (1 session x 90 = 90 samples)
            X_gen_tr = np.vstack([data_dict[(target_sub, ses)] for ses in tr_sessions])
            X_gen_te = data_dict[(target_sub, te_session)]
            gen_tr_count = len(X_gen_tr)
            gen_te_count = len(X_gen_te)
            
            # Impostor training samples from other subjects in tr_sessions
            other_subs = [s for s in SUBJECTS if s != target_sub]
            n_imp_tr_per_sub = len(X_gen_tr) // len(other_subs)
            X_imp_tr_list = []
            for s in other_subs:
                s_tr = np.vstack([data_dict[(s, SUBJECT_SESSIONS[s][i])] for i in tr_ses_idx_list])
                X_imp_tr_list.append(s_tr[:n_imp_tr_per_sub])
            X_imp_tr = np.vstack(X_imp_tr_list)
            
            # Impostor test samples from other subjects in te_session (balanced: exactly 90 samples)
            n_imp_te_per_sub = len(X_gen_te) // len(other_subs)
            X_imp_te = np.vstack([
                data_dict[(s, SUBJECT_SESSIONS[s][te_ses_idx])][:n_imp_te_per_sub]
                for s in other_subs
            ])
            imp_te_count = len(X_imp_te)
            
            # Combine train and test splits
            X_train = np.vstack([X_gen_tr, X_imp_tr])
            y_train = np.concatenate([np.ones(len(X_gen_tr)), np.zeros(len(X_imp_tr))])
            
            X_test = np.vstack([X_gen_te, X_imp_te])
            y_test = np.concatenate([np.ones(len(X_gen_te)), np.zeros(len(X_imp_te))])
            
            # Scaler strictly on train split
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            for model_name in ALL_MODEL_NAMES:
                clf = get_model_pipeline(model_name, in_features=X_train_scaled.shape[1])
                clf.fit(X_train_scaled, y_train)
                probs = clf.predict_proba(X_test_scaled)[:, 1]
                res = evaluate_biometric_predictions(y_test, probs)
                fold_metrics[model_name].append(res)
                
        tr_label = f"Sessions {[i + 1 for i in tr_ses_idx_list]}"
        te_label = f"Session {te_ses_idx + 1}"
        
        for model_name, metric_list in fold_metrics.items():
            expA_results.append({
                "Experiment": "Multi-Session (Train 3, Test 1)",
                "Region": region_name,
                "Model": model_name,
                "Train_Sessions": tr_label,
                "Test_Session": te_label,
                "N_genuine_train": gen_tr_count,
                "N_genuine_test": gen_te_count,
                "N_impostor_test": imp_te_count,
                "Accuracy": float(np.mean([m["Accuracy"] for m in metric_list])),
                "EER": float(np.mean([m["EER"] for m in metric_list])),
                "FAR": float(np.mean([m["FAR"] for m in metric_list])),
                "FRR": float(np.mean([m["FRR"] for m in metric_list])),
                "FP": float(np.mean([m["FP"] for m in metric_list])),
                "FN": float(np.mean([m["FN"] for m in metric_list]))
            })

df_expA = pd.DataFrame(expA_results)
format_dict = {
    "Accuracy": "{:.4f}",
    "EER": "{:.4f}",
    "FAR": "{:.4f}",
    "FRR": "{:.4f}",
    "FP": "{:.1f}",
    "FN": "{:.1f}"
}

print("\n" + "=" * 110)
print("EXPERIMENT A: MULTI-SESSION LOSO RESULTS (AVERAGED ACROSS ALL 4 HELD-OUT ROTATIONS):")
print("=" * 110)
summary_expA = df_expA.groupby(["Region", "Model"])[["Accuracy", "EER", "FAR", "FRR", "FP", "FN"]].mean().reset_index()
display(summary_expA.sort_values(by=["Region", "EER"]).style.format(format_dict))


Beginning Experiment A: Multi-Session Training (LOSO) across 6 regions and 8 models...

--- Experiment A: Region Frontal (14 channels, 280 spectral features) ---


--- Experiment A: Region Central (14 channels, 280 spectral features) ---


--- Experiment A: Region Temporal (6 channels, 120 spectral features) ---


--- Experiment A: Region Parietal (14 channels, 280 spectral features) ---


--- Experiment A: Region Occipital (10 channels, 200 spectral features) ---


--- Experiment A: Region ALL (58 channels, 1160 spectral features) ---



EXPERIMENT A: MULTI-SESSION LOSO RESULTS (AVERAGED ACROSS ALL 4 HELD-OUT ROTATIONS):


,Region,Model,Accuracy,EER,FAR,FRR,FP,FN
5,ALL,SVM_RBF,0.9522,0.0385,0.0455,0.0500,4.0,4.5
4,ALL,Res-MLP,0.9419,0.0446,0.0511,0.0650,4.5,5.8
2,ALL,ExtraTrees,0.9272,0.0522,0.0795,0.0661,7.0,6.0
7,ALL,XGBoost,0.9076,0.0635,0.0750,0.1094,6.6,9.9
3,ALL,LightGBM,0.9225,0.0724,0.0761,0.0789,6.7,7.1
6,ALL,SpecFormer,0.8278,0.1677,0.1733,0.1711,15.2,15.4
1,ALL,BiLSTM-Attention,0.7809,0.2273,0.2722,0.1672,23.9,15.1
0,ALL,1D-CNN,0.6674,0.2770,0.3125,0.3522,27.5,31.7
13,Central,SVM_RBF,0.9292,0.0612,0.0767,0.0650,6.8,5.8
11,Central,LightGBM,0.9126,0.0702,0.1000,0.0750,8.8,6.8


## Section 4: Experiment B – Single-Session Training and Remaining-Session Testing

We implement the complementary single-session training experiment:
- Train on Session 1 $\to$ Test on Sessions 2, 3, 4
- Train on Session 2 $\to$ Test on Sessions 1, 3, 4
- Train on Session 3 $\to$ Test on Sessions 1, 2, 4
- Train on Session 4 $\to$ Test on Sessions 1, 2, 3

### Balanced Evaluation:
- Genuine training samples: 90 epochs from single training session.
- Genuine test samples: 270 epochs from the 3 remaining sessions.
- Impostor test samples: 270 epochs from other subjects across those 3 sessions ($N_{impostor} = N_{genuine} = 270$).


In [5]:
# Execute Experiment B: Single-Session Training against Remaining Sessions
SINGLE_VS_REST_FOLDS = [
    (0, [1, 2, 3]),
    (1, [0, 2, 3]),
    (2, [0, 1, 3]),
    (3, [0, 1, 2])
]

expB_results = []
print(f"Beginning Experiment B: Single-Session Training across {len(REGIONS)} regions and {len(ALL_MODEL_NAMES)} models...\n")

for region_name, ch_list in REGIONS.items():
    print(f"--- Experiment B: Region {region_name} ({len(ch_list)} channels, {len(ch_list)*20} spectral features) ---")
    data_dict = get_session_features(ch_list)
    
    for fold_idx, (tr_ses_idx, te_ses_idx_list) in enumerate(SINGLE_VS_REST_FOLDS, 1):
        fold_metrics = {m: [] for m in ALL_MODEL_NAMES}
        gen_tr_count = 0
        gen_te_count = 0
        imp_te_count = 0
        
        for target_sub in SUBJECTS:
            sub_sessions = SUBJECT_SESSIONS[target_sub]
            tr_session = sub_sessions[tr_ses_idx]
            te_sessions = [sub_sessions[i] for i in te_ses_idx_list]
            
            # Genuine train (90 samples) & test (270 samples)
            X_gen_tr = data_dict[(target_sub, tr_session)]
            X_gen_te = np.vstack([data_dict[(target_sub, ses)] for ses in te_sessions])
            gen_tr_count = len(X_gen_tr)
            gen_te_count = len(X_gen_te)
            
            # Impostor train (90 samples) from other subjects
            other_subs = [s for s in SUBJECTS if s != target_sub]
            n_imp_tr_per_sub = len(X_gen_tr) // len(other_subs)
            X_imp_tr = np.vstack([
                data_dict[(s, SUBJECT_SESSIONS[s][tr_ses_idx])][:n_imp_tr_per_sub]
                for s in other_subs
            ])
            
            # Impostor test (270 samples) from other subjects across te_sessions
            n_imp_te_per_sub = len(X_gen_te) // len(other_subs)
            X_imp_te_list = []
            for s in other_subs:
                s_te = np.vstack([data_dict[(s, SUBJECT_SESSIONS[s][i])] for i in te_ses_idx_list])
                X_imp_te_list.append(s_te[:n_imp_te_per_sub])
            X_imp_te = np.vstack(X_imp_te_list)
            imp_te_count = len(X_imp_te)
            
            X_train = np.vstack([X_gen_tr, X_imp_tr])
            y_train = np.concatenate([np.ones(len(X_gen_tr)), np.zeros(len(X_imp_tr))])
            
            X_test = np.vstack([X_gen_te, X_imp_te])
            y_test = np.concatenate([np.ones(len(X_gen_te)), np.zeros(len(X_imp_te))])
            
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            for model_name in ALL_MODEL_NAMES:
                clf = get_model_pipeline(model_name, in_features=X_train_scaled.shape[1])
                clf.fit(X_train_scaled, y_train)
                probs = clf.predict_proba(X_test_scaled)[:, 1]
                res = evaluate_biometric_predictions(y_test, probs)
                fold_metrics[model_name].append(res)
                
        tr_label = f"Session {tr_ses_idx + 1}"
        te_label = f"Sessions {[i + 1 for i in te_ses_idx_list]}"
        
        for model_name, metric_list in fold_metrics.items():
            expB_results.append({
                "Experiment": "Single-Session (Train 1, Test 3)",
                "Region": region_name,
                "Model": model_name,
                "Train_Sessions": tr_label,
                "Test_Session": te_label,
                "N_genuine_train": gen_tr_count,
                "N_genuine_test": gen_te_count,
                "N_impostor_test": imp_te_count,
                "Accuracy": float(np.mean([m["Accuracy"] for m in metric_list])),
                "EER": float(np.mean([m["EER"] for m in metric_list])),
                "FAR": float(np.mean([m["FAR"] for m in metric_list])),
                "FRR": float(np.mean([m["FRR"] for m in metric_list])),
                "FP": float(np.mean([m["FP"] for m in metric_list])),
                "FN": float(np.mean([m["FN"] for m in metric_list]))
            })

df_expB = pd.DataFrame(expB_results)
print("\n" + "=" * 110)
print("EXPERIMENT B: SINGLE-SESSION TRAINING RESULTS (AVERAGED ACROSS ALL 4 TRAINING SESSIONS):")
print("=" * 110)
summary_expB = df_expB.groupby(["Region", "Model"])[["Accuracy", "EER", "FAR", "FRR", "FP", "FN"]].mean().reset_index()
display(summary_expB.sort_values(by=["Region", "EER"]).style.format(format_dict))


Beginning Experiment B: Single-Session Training across 6 regions and 8 models...

--- Experiment B: Region Frontal (14 channels, 280 spectral features) ---


--- Experiment B: Region Central (14 channels, 280 spectral features) ---


--- Experiment B: Region Temporal (6 channels, 120 spectral features) ---


--- Experiment B: Region Parietal (14 channels, 280 spectral features) ---


--- Experiment B: Region Occipital (10 channels, 200 spectral features) ---


--- Experiment B: Region ALL (58 channels, 1160 spectral features) ---



EXPERIMENT B: SINGLE-SESSION TRAINING RESULTS (AVERAGED ACROSS ALL 4 TRAINING SESSIONS):


,Region,Model,Accuracy,EER,FAR,FRR,FP,FN
5,ALL,SVM_RBF,0.8799,0.1023,0.0890,0.1509,23.9,40.8
4,ALL,Res-MLP,0.8704,0.1220,0.1297,0.1294,34.8,35.0
2,ALL,ExtraTrees,0.8633,0.1299,0.1224,0.1509,32.8,40.8
3,ALL,LightGBM,0.8201,0.1705,0.1336,0.2259,35.8,61.0
7,ALL,XGBoost,0.8072,0.1734,0.1567,0.2287,42.0,61.8
6,ALL,SpecFormer,0.7845,0.2138,0.2250,0.2061,60.3,55.6
1,ALL,BiLSTM-Attention,0.7676,0.2348,0.2750,0.1902,73.7,51.4
0,ALL,1D-CNN,0.6678,0.2844,0.3369,0.3274,90.3,88.4
13,Central,SVM_RBF,0.8620,0.1224,0.1144,0.1615,30.6,43.6
10,Central,ExtraTrees,0.8544,0.1365,0.1446,0.1467,38.8,39.6


## Section 5: Direct Comparison: Multi-Session vs Single-Session Training

We perform a direct side-by-side comparative analysis between:
- **Multi-Session Enrollment (Train on 3 Sessions)**
- **Single-Session Enrollment (Train on 1 Session)**

This comparison directly quantifies how enrolling users across multiple recording sessions captures intra-subject spectral variability, reduces template drift, and enhances classification accuracy.


In [6]:
# Side-by-Side Comparison: Multi-Session (Train 3) vs Single-Session (Train 1)
comp_df = pd.merge(
    summary_expB[["Region", "Model", "Accuracy", "EER", "FAR", "FRR"]],
    summary_expA[["Region", "Model", "Accuracy", "EER", "FAR", "FRR"]],
    on=["Region", "Model"],
    suffixes=("_SingleSession", "_MultiSession")
)

# Relative EER reduction achieved by multi-session training
comp_df["EER_Reduction_Pct"] = (
    (comp_df["EER_SingleSession"] - comp_df["EER_MultiSession"]) / (comp_df["EER_SingleSession"] + 1e-12) * 100.0
)
comp_df["Accuracy_Gain_Pct"] = (
    (comp_df["Accuracy_MultiSession"] - comp_df["Accuracy_SingleSession"]) * 100.0
)

print("=" * 120)
print("DIRECT COMPARISON: SINGLE-SESSION VS MULTI-SESSION TRAINING (GLOBAL ALL MONTAGE):")
print("=" * 120)
display_cols = [
    "Model",
    "Accuracy_SingleSession", "Accuracy_MultiSession", "Accuracy_Gain_Pct",
    "EER_SingleSession", "EER_MultiSession", "EER_Reduction_Pct",
    "FAR_SingleSession", "FAR_MultiSession",
    "FRR_SingleSession", "FRR_MultiSession"
]
all_comp = comp_df[comp_df["Region"] == "ALL"].sort_values(by="EER_MultiSession")[display_cols]

comp_format = {
    "Accuracy_SingleSession": "{:.4f}",
    "Accuracy_MultiSession": "{:.4f}",
    "Accuracy_Gain_Pct": "{:+.2f}%",
    "EER_SingleSession": "{:.4f}",
    "EER_MultiSession": "{:.4f}",
    "EER_Reduction_Pct": "{:+.2f}%",
    "FAR_SingleSession": "{:.4f}",
    "FAR_MultiSession": "{:.4f}",
    "FRR_SingleSession": "{:.4f}",
    "FRR_MultiSession": "{:.4f}"
}
display(all_comp.style.format(comp_format))

print("\n" + "=" * 120)
print("REGIONAL MEAN COMPARISON (AVERAGED ACROSS ALL 8 MODELS):")
print("=" * 120)
reg_comp = comp_df.groupby("Region")[[
    "Accuracy_SingleSession", "Accuracy_MultiSession", "Accuracy_Gain_Pct",
    "EER_SingleSession", "EER_MultiSession", "EER_Reduction_Pct",
    "FAR_SingleSession", "FAR_MultiSession",
    "FRR_SingleSession", "FRR_MultiSession"
]].mean().reset_index().sort_values(by="EER_MultiSession")
display(reg_comp.style.format(comp_format))

# Summary Findings
best_multi = summary_expA.loc[summary_expA["EER"].idxmin()]
print("\n" + "=" * 80)
print("KEY MULTI-SESSION BIOMETRIC FINDINGS:")
print("=" * 80)
print(f"Top Multi-Session System : Region: {best_multi['Region']} | Model: {best_multi['Model']}")
print(f"  * Multi-Session EER     : {best_multi['EER']:.4f}")
print(f"  * Multi-Session Accuracy: {best_multi['Accuracy']:.4f}")
print(f"  * Multi-Session FAR     : {best_multi['FAR']:.4f}")
print(f"  * Multi-Session FRR     : {best_multi['FRR']:.4f}")
print(f"Average EER Reduction across regions with multi-session enrollment: {comp_df['EER_Reduction_Pct'].mean():.2f}%")
print("Notebook 4 multi-session evaluation completed successfully.")


DIRECT COMPARISON: SINGLE-SESSION VS MULTI-SESSION TRAINING (GLOBAL ALL MONTAGE):


,Model,Accuracy_SingleSession,Accuracy_MultiSession,Accuracy_Gain_Pct,EER_SingleSession,EER_MultiSession,EER_Reduction_Pct,FAR_SingleSession,FAR_MultiSession,FRR_SingleSession,FRR_MultiSession
5,SVM_RBF,0.8799,0.9522,+7.23%,0.1023,0.0385,+62.40%,0.0890,0.0455,0.1509,0.0500
4,Res-MLP,0.8704,0.9419,+7.14%,0.1220,0.0446,+63.41%,0.1297,0.0511,0.1294,0.0650
2,ExtraTrees,0.8633,0.9272,+6.40%,0.1299,0.0522,+59.82%,0.1224,0.0795,0.1509,0.0661
7,XGBoost,0.8072,0.9076,+10.04%,0.1734,0.0635,+63.39%,0.1567,0.0750,0.2287,0.1094
3,LightGBM,0.8201,0.9225,+10.24%,0.1705,0.0724,+57.50%,0.1336,0.0761,0.2259,0.0789
6,SpecFormer,0.7845,0.8278,+4.33%,0.2138,0.1677,+21.58%,0.2250,0.1733,0.2061,0.1711
1,BiLSTM-Attention,0.7676,0.7809,+1.33%,0.2348,0.2273,+3.23%,0.2750,0.2722,0.1902,0.1672
0,1D-CNN,0.6678,0.6674,-0.04%,0.2844,0.2770,+2.61%,0.3369,0.3125,0.3274,0.3522



REGIONAL MEAN COMPARISON (AVERAGED ACROSS ALL 8 MODELS):


,Region,Accuracy_SingleSession,Accuracy_MultiSession,Accuracy_Gain_Pct,EER_SingleSession,EER_MultiSession,EER_Reduction_Pct,FAR_SingleSession,FAR_MultiSession,FRR_SingleSession,FRR_MultiSession
0,ALL,0.8076,0.8659,+5.83%,0.1789,0.1179,+41.74%,0.1835,0.1357,0.2012,0.1325
4,Parietal,0.8192,0.8607,+4.15%,0.1733,0.1242,+33.01%,0.1841,0.1484,0.1774,0.1303
1,Central,0.8018,0.8502,+4.85%,0.1883,0.1319,+35.85%,0.1988,0.1708,0.1977,0.1292
5,Temporal,0.7822,0.8237,+4.15%,0.2071,0.1554,+26.65%,0.1983,0.2021,0.2372,0.1511
3,Occipital,0.7762,0.8302,+5.39%,0.2193,0.1584,+31.38%,0.2197,0.1684,0.2278,0.1712
2,Frontal,0.7818,0.8267,+4.49%,0.2111,0.1594,+26.95%,0.2075,0.1756,0.2288,0.1710



KEY MULTI-SESSION BIOMETRIC FINDINGS:
Top Multi-Session System : Region: ALL | Model: SVM_RBF
  * Multi-Session EER     : 0.0385
  * Multi-Session Accuracy: 0.9522
  * Multi-Session FAR     : 0.0455
  * Multi-Session FRR     : 0.0500
Average EER Reduction across regions with multi-session enrollment: 32.60%
Notebook 4 multi-session evaluation completed successfully.
